# Baymax — Module 3: workout-calorie model (Gradient Boosting)

**What this notebook does:** cleans the Kaggle *Gym Members Exercise Dataset*, checks whether its calorie values agree with
published energy costs, trains a Gradient Boosting model, and decides whether that model is safe to put in the app.

**What the model predicts:** calories burned in **one workout session**, from age, sex, weight, height, session length,
experience level and workout type (no heart rate — the app has no heart-rate sensor). It does **not** choose or recommend
exercises; exercise selection in the app is rule-based.

**How to use it on Kaggle:** *Add Input* → search **"Gym Members Exercise Dataset"** (by valakhorasani) → *Run All*.
No GPU and no internet are needed. Results are written to `/kaggle/working`.

> **Expect this outcome:** the dataset is tidy but its calorie values are not physiologically realistic (see section 3), so the safety
> check in section 6 will **refuse to export** a model. That is the intended behaviour; the notebook documents why.

In [ ]:
import os, json, glob, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split

warnings.filterwarnings('ignore')
try:
    display  # available inside Jupyter / Kaggle
except NameError:
    display = print

INPUT_DIR = os.environ.get('GYM_INPUT_DIR', '/kaggle/input')
OUT_DIR = os.environ.get('GYM_OUT_DIR', '/kaggle/working')
os.makedirs(OUT_DIR, exist_ok=True)

# 2024 Adult Compendium of Physical Activities (pacompendium.com). MET = kcal per kg per hour.
COMPENDIUM_MET = {'Yoga': 2.3,       # 02175 Yoga, general
                  'Strength': 3.5,   # 02054 Resistance training, multiple exercises
                  'Cardio': 7.3,     # 02000 Aerobic, general
                  'HIIT': 7.5}       # 02040 Circuit training, vigorous intensity
TYPES = ['Cardio', 'HIIT', 'Strength', 'Yoga']
PLAUSIBLE_RATIO = (0.5, 2.0)        # "realistic" = within 0.5x-2x of the Compendium
print('input folder:', INPUT_DIR, '| output folder:', OUT_DIR)

## 1. Load the data
The notebook looks for the dataset's CSV anywhere under `/kaggle/input` (it also accepts the cleaned file from the Baymax zip).

In [ ]:
RENAME = {
    'Age': 'age', 'Gender': 'gender', 'Weight (kg)': 'weight_kg', 'Height (m)': 'height_m',
    'Max_BPM': 'max_bpm', 'Avg_BPM': 'avg_bpm', 'Resting_BPM': 'resting_bpm',
    'Session_Duration (hours)': 'duration_h', 'Calories_Burned': 'calories_burned',
    'Workout_Type': 'workout_type', 'Fat_Percentage': 'fat_percentage',
    'Water_Intake (liters)': 'water_l', 'Workout_Frequency (days/week)': 'workout_days_per_week',
    'Experience_Level': 'experience_level', 'BMI': 'bmi',
}
REQUIRED = ['age', 'gender', 'weight_kg', 'height_m', 'max_bpm', 'avg_bpm', 'resting_bpm', 'duration_h',
            'calories_burned', 'workout_type', 'experience_level', 'bmi']

csv_path = None
for path in sorted(glob.glob(os.path.join(INPUT_DIR, '**', '*.csv'), recursive=True)):
    head = pd.read_csv(path, nrows=2)
    cols = {RENAME.get(c.strip(), c.strip()) for c in head.columns}
    if set(REQUIRED) <= cols:
        csv_path = path
        break
if csv_path is None:
    raise SystemExit('No gym-members CSV found. Use Add Input -> "Gym Members Exercise Dataset".')

raw = pd.read_csv(csv_path)
df = raw.rename(columns=lambda c: RENAME.get(c.strip(), c.strip())).copy()
print('file:', csv_path)
print('shape:', raw.shape)
display(raw.head())

## 2. Clean the data
Only **physically impossible** rows are removed. Nothing is imputed, rescaled or invented.

In [ ]:
steps = []
n0 = len(df)

for c in ['gender', 'workout_type']:
    before = df[c].copy()
    df[c] = df[c].astype(str).str.strip().str.title().replace({'Hiit': 'HIIT'})
    steps.append((f'whitespace / capitalisation in {c}', int((before != df[c]).sum()), 'standardised'))

steps.append(('missing values', int(df[REQUIRED].isna().any(axis=1).sum()), 'rows removed if any'))
df = df.dropna(subset=REQUIRED)

dup = int(df.duplicated().sum())
steps.append(('exact duplicate rows', dup, 'removed' if dup else 'none found'))
df = df.drop_duplicates()

bad = df['avg_bpm'] > df['max_bpm']
steps.append(('average heart rate above the maximum (impossible)', int(bad.sum()), 'removed'))
df = df[~bad]

bad = df['avg_bpm'] < df['resting_bpm']
steps.append(('average heart rate below resting heart rate', int(bad.sum()), 'removed' if bad.any() else 'none found'))
df = df[~bad]

bad = (df['calories_burned'] <= 0) | (df['duration_h'] <= 0) | (df['weight_kg'] <= 0) | (df['height_m'] <= 0)
steps.append(('zero/negative calories, duration, weight or height', int(bad.sum()), 'removed' if bad.any() else 'none found'))
df = df[~bad]

bad = (df['weight_kg'] / df['height_m'] ** 2 - df['bmi']).abs() > 0.5
steps.append(('BMI disagrees with weight / height^2 by > 0.5', int(bad.sum()), 'removed' if bad.any() else 'none found'))
df = df[~bad]

q1, q3 = df['calories_burned'].quantile([.25, .75])
out = (df['calories_burned'] < q1 - 1.5 * (q3 - q1)) | (df['calories_burned'] > q3 + 1.5 * (q3 - q1))
steps.append(('calories outside 1.5 x IQR', int(out.sum()), 'kept (plausible long sessions)'))

df = df.reset_index(drop=True)

# helper columns (derived; originals are untouched)
df['kcal_per_hour'] = (df['calories_burned'] / df['duration_h']).round(1)
df['implied_met'] = (df['calories_burned'] / df['duration_h'] / df['weight_kg']).round(2)   # kcal per kg per hour == MET
df['compendium_met'] = df['workout_type'].map(COMPENDIUM_MET)
df['met_ratio'] = (df['implied_met'] / df['compendium_met']).round(2)
df['plausible_energy'] = df['met_ratio'].between(*PLAUSIBLE_RATIO)

display(pd.DataFrame(steps, columns=['check', 'rows affected', 'action']))
print(f'rows: {n0} raw -> {len(df)} clean')
df.to_csv(os.path.join(OUT_DIR, 'gym_members_clean.csv'), index=False)

## 3. Does the data agree with published energy costs?
`implied_met` = calories ÷ hours ÷ body weight. That is exactly what a MET value measures, so it can be compared with the
2024 Compendium of Physical Activities. In real measurements the type of exercise is the main driver of energy cost: yoga ≈ 2.3 METs,
HIIT ≈ 7.5 METs.

In [ ]:
by_type = df.groupby('workout_type').agg(
    sessions=('implied_met', 'size'),
    median_met_in_data=('implied_met', 'median'),
    compendium_met=('compendium_met', 'first'),
    realistic_sessions=('plausible_energy', 'sum')).round(2)
by_type['data / compendium'] = (by_type['median_met_in_data'] / by_type['compendium_met']).round(2)
display(by_type)

# one-way ANOVA: how much does workout type explain?
grand = df['implied_met'].mean()
groups = [g['implied_met'].values for _, g in df.groupby('workout_type')]
ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
ssw = sum(((g - g.mean()) ** 2).sum() for g in groups)
F = (ssb / (len(groups) - 1)) / (ssw / (len(df) - len(groups)))
print(f'Workout type explains {100 * ssb / (ssb + ssw):.2f}% of the variation in implied MET (ANOVA F = {F:.2f}).')
print(f'Sessions with a realistic energy cost: {df["plausible_energy"].mean() * 100:.0f}% overall; '
      f'yoga {int(by_type.loc["Yoga", "realistic_sessions"])} of {int(by_type.loc["Yoga", "sessions"])}.')

print('\nExperience level vs training days per week (a perfectly clean rule is typical of generated data):')
display(pd.crosstab(df['experience_level'], df['workout_days_per_week']))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
order = ['Yoga', 'Strength', 'Cardio', 'HIIT']
ax[0].boxplot([df.loc[df.workout_type == t, 'implied_met'] for t in order], showfliers=False)
ax[0].set_xticklabels(order)
ax[0].scatter(range(1, 5), [COMPENDIUM_MET[t] for t in order], color='red', zorder=3, s=70, label='Compendium (published)')
ax[0].set_ylabel('implied MET (kcal per kg per hour)'); ax[0].set_title('Energy cost per workout type: data vs published'); ax[0].legend()
for t in order:
    s = df[df.workout_type == t]
    ax[1].scatter(s['duration_h'], s['calories_burned'], s=10, alpha=.5, label=t)
ax[1].set_xlabel('session length (hours)'); ax[1].set_ylabel('calories burned')
ax[1].set_title('Calories vs duration: all workout types overlap'); ax[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'data_vs_compendium.png'), dpi=120)
plt.show()

## 4. Train and compare
Inputs are only what the Baymax app knows. The baseline is the app's current method: **MET × body weight × hours** from the Compendium.

In [ ]:
FEATURES = ['age', 'sex_male', 'weight_kg', 'height_m', 'duration_h', 'experience',
            'is_cardio', 'is_hiit', 'is_strength', 'is_yoga']

def to_features(frame):
    X = pd.DataFrame({
        'age': frame['age'], 'sex_male': (frame['gender'] == 'Male').astype(int),
        'weight_kg': frame['weight_kg'], 'height_m': frame['height_m'],
        'duration_h': frame['duration_h'], 'experience': frame['experience_level'],
        **{f'is_{t.lower()}': (frame['workout_type'] == t).astype(int) for t in TYPES}})
    return X[FEATURES]

X = to_features(df)
y = df['calories_burned'].astype(float)
baseline = df['workout_type'].map(COMPENDIUM_MET) * df['weight_kg'] * df['duration_h']

X_tr, X_te, y_tr, y_te, b_tr, b_te = train_test_split(X, y, baseline, test_size=0.2, random_state=42)
params = dict(n_estimators=250, learning_rate=0.05, max_depth=3, subsample=0.8, min_samples_leaf=5, random_state=42)
gb = GradientBoostingRegressor(**params).fit(X_tr, y_tr)
lin = LinearRegression().fit(X_tr, y_tr)

def scores(name, truth, pred):
    return {'model': name, 'MAE (kcal)': round(mean_absolute_error(truth, pred), 1),
            'RMSE (kcal)': round(mean_squared_error(truth, pred) ** 0.5, 1), 'R2': round(r2_score(truth, pred), 3)}

results = pd.DataFrame([
    scores('Compendium MET x kg x hours (app today)', y_te, b_te),
    scores('Linear regression', y_te, lin.predict(X_te)),
    scores('Gradient Boosting', y_te, gb.predict(X_te))])
display(results)

cv = cross_val_predict(GradientBoostingRegressor(**params), X, y, cv=KFold(5, shuffle=True, random_state=1))
print('5-fold cross-validation (Gradient Boosting):', scores('cv', y, cv))
imp = pd.Series(gb.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nfeature importance:'); display(imp.round(3).head(6).to_frame('importance'))
beats = scores('gb', y_te, gb.predict(X_te))['MAE (kcal)'] < scores('met', y_te, b_te)['MAE (kcal)']
print('More accurate than the MET estimate on this data:', beats)

## 5. Why a low error is not enough
The Gradient Boosting model fits this dataset very well — but only because it learned the dataset's quirks. Look at what it believes about a
standard person (30-year-old man, 70 kg) doing **one hour** of each workout, next to the published energy cost.

In [ ]:
def standard_person(weight=70.0, wtype='Cardio'):
    return pd.DataFrame([{'age': 30, 'sex_male': 1, 'weight_kg': weight, 'height_m': 1.75, 'duration_h': 1.0, 'experience': 2,
                          **{f'is_{t.lower()}': int(t == wtype) for t in TYPES}}])[FEATURES]

rows, realistic = [], True
for t in TYPES:
    pred = float(gb.predict(standard_person(70, t))[0])
    expected = COMPENDIUM_MET[t] * 70
    ratio = pred / expected
    ok = PLAUSIBLE_RATIO[0] <= ratio <= PLAUSIBLE_RATIO[1]
    realistic &= ok
    rows.append({'workout (1 h, 70 kg man)': t, 'model says (kcal)': round(pred), 'published (kcal)': round(expected),
                 'model / published': round(ratio, 2), 'realistic (0.5x-2x)': 'yes' if ok else 'NO'})
display(pd.DataFrame(rows))

heavier = float(gb.predict(standard_person(95, 'Cardio'))[0]) > float(gb.predict(standard_person(55, 'Cardio'))[0])
print('Does a 95 kg person burn more than a 55 kg person in the same session?', heavier)
physiology_ok = bool(realistic and heavier)

## 6. Safety gate and export
A model is exported for the app **only if** it (1) beats the MET estimate **and** (2) agrees with published energy costs and rises with body weight.
Accuracy on a dataset alone must never push a physiologically wrong model into a health app.

In [ ]:
export = bool(beats and physiology_ok)
report = {
    'trained_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'rows_after_cleaning': int(len(df)),
    'cleaning_steps': [dict(check=a, rows=b, action=c) for a, b, c in steps],
    'test_metrics': results.to_dict(orient='records'),
    'more_accurate_than_met': bool(beats),
    'physiology_ok': physiology_ok,
    'workout_type_explains_variation_percent': round(100 * ssb / (ssb + ssw), 2),
    'exported': export,
}

def tree_to_dict(est):
    t = est.tree_
    return {'left': t.children_left.tolist(), 'right': t.children_right.tolist(), 'feature': t.feature.tolist(),
            'threshold': [float(v) for v in t.threshold], 'value': [float(v) for v in t.value[:, 0, 0]]}

if export:
    model = {'name': 'calorie_gb', 'version': datetime.now(timezone.utc).strftime('%Y%m%d'),
             'features': FEATURES, 'workout_types': TYPES, 'learning_rate': gb.learning_rate,
             'init': float(gb.init_.constant_[0][0]), 'trees': [tree_to_dict(e[0]) for e in gb.estimators_],
             'baseline_mets': COMPENDIUM_MET}
    idx = np.random.RandomState(3).choice(len(X_te), 10, replace=False)
    model['test_vectors'] = [{'x': [float(v) for v in X_te.iloc[i]], 'kcal': float(gb.predict(X_te.iloc[[i]])[0])} for i in idx]
    with open(os.path.join(OUT_DIR, 'calorie_gb.json'), 'w') as f:
        json.dump(model, f)
    print('EXPORTED calorie_gb.json -> copy it into the app at assets/models/')
else:
    why = []
    if not beats:
        why.append('it is not more accurate than the Compendium estimate')
    if not physiology_ok:
        why.append('it contradicts published energy costs (the training data does, see section 3)')
    print('NOT EXPORTED: ' + ' and '.join(why) + '.')
    print('The app keeps using the Compendium MET estimate for workout calories.')

with open(os.path.join(OUT_DIR, 'calorie_model_report.json'), 'w') as f:
    json.dump(report, f, indent=1)
print('files in', OUT_DIR, ':', sorted(os.listdir(OUT_DIR)))

## 7. What to do next
* Use this notebook's output (the tables and the two charts) in your report as evidence of dataset validation.
* A legitimate model needs data with **measured** energy expenditure (indirect calorimetry or doubly-labelled water). The IAEA
  doubly-labelled-water database is the authentic source for *daily* energy expenditure and is available by application:
  https://doubly-labelled-water-database.iaea.org
* Until then the app uses the peer-reviewed Compendium values, which is the sound choice.